# Fase 4 y Fase 5: KPIs y dashboard minimo

Este notebook deja un catalogo de KPIs alineado con el PDF:

- KPIs de negocio justificados.
- Relacion explicita con las hipotesis.
- Interpretacion ejecutiva y soporte en el dashboard.
- 5 visualizaciones para apoyar el dashboard.
- Validacion del archivo `dashboard/hr_dashboard.pbix`.


In [ ]:
# Importamos las librerias minimas del notebook.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Definimos rutas cortas para datos y dashboard.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DASHBOARD_FILE = PROJECT_ROOT / "dashboard" / "hr_dashboard.pbix"
VIS_DIR = PROJECT_ROOT / "dashboard" / "visualizaciones"

# Activamos un estilo simple para los graficos.
sns.set_theme(style="whitegrid")
VIS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Cargamos la base limpia que ya viene del notebook 01.
df = pd.read_csv(DATA_DIR / "clean_hr_analytics.csv")

# Confirmamos que el archivo se cargo bien.
print(f"Registros: {len(df):,}")
print(f"Columnas: {df.shape[1]}")
display(df.head())

## KPIs de negocio alineados con las hipotesis


In [ ]:
# Calculamos KPIs de negocio que se puedan defender desde el problema de retencion.
turnover_general = round(float(df["left"].mean() * 100), 2)
satisfaccion_promedio = round(float(df["satisfaction_level"].mean()), 3)
horas_promedio = round(float(df["average_monthly_hours"].mean()), 1)

rotacion_sat_baja_raw = float(df.loc[df["satisfaction_category"] == "Baja", "left"].mean() * 100)
rotacion_sat_alta_raw = float(df.loc[df["satisfaction_category"] == "Alta", "left"].mean() * 100)
rotacion_sat_baja = round(rotacion_sat_baja_raw, 2)
brecha_rotacion_satisfaccion = round(rotacion_sat_baja_raw - rotacion_sat_alta_raw, 2)

rotacion_salario_bajo_raw = float(df.loc[df["salary_label"] == "Bajo", "left"].mean() * 100)
rotacion_salario_alto_raw = float(df.loc[df["salary_label"] == "Alto", "left"].mean() * 100)
rotacion_salario_bajo = round(rotacion_salario_bajo_raw, 2)
brecha_salario = round(rotacion_salario_bajo_raw - rotacion_salario_alto_raw, 2)

rotacion_alta_carga = round(float(df.loc[df["workload_category"] == "Alta carga", "left"].mean() * 100), 2)

print("Tasa de rotacion general:", turnover_general)
print("Satisfaccion promedio:", satisfaccion_promedio)
print("Horas mensuales promedio:", horas_promedio)
print("Rotacion con satisfaccion baja:", rotacion_sat_baja)
print("Brecha de rotacion por satisfaccion:", brecha_rotacion_satisfaccion)
print("Rotacion con salario bajo:", rotacion_salario_bajo)
print("Brecha salario bajo vs alto:", brecha_salario)
print("Rotacion en alta carga laboral:", rotacion_alta_carga)


In [ ]:
# Armamos la tabla final de KPIs con justificacion, relacion con hipotesis y soporte en el dashboard.
kpis = pd.DataFrame(
    [
        [
            "Tasa de rotacion general",
            turnover_general,
            "%",
            "Resume el nivel total de salida y sirve para seguir el riesgo global de retencion.",
            "H2, H3 y H4.",
            "Casi una cuarta parte de los empleados abandono la empresa.",
            "Tarjeta ejecutiva de tasa de rotacion.",
        ],
        [
            "Satisfaccion promedio",
            satisfaccion_promedio,
            "indice 0-1",
            "Monitorea el clima laboral agregado que puede anticipar deterioro en permanencia.",
            "H1 y H2.",
            "El nivel medio de satisfaccion es moderado y debe vigilarse junto con la rotacion.",
            "Tarjeta ejecutiva y dispersion horas vs satisfaccion.",
        ],
        [
            "Horas mensuales promedio",
            horas_promedio,
            "horas",
            "Controla la carga laboral promedio como variable operativa asociada al desgaste.",
            "H1 y H4.",
            "La carga mensual promedio es alta y conviene leerla como variable de contexto laboral.",
            "Tarjeta ejecutiva, dispersion horas vs satisfaccion y visuales por departamento.",
        ],
        [
            "Rotacion con satisfaccion baja",
            rotacion_sat_baja,
            "%",
            "Identifica el segmento prioritario para acciones de clima y acompanamiento.",
            "H2.",
            "Los empleados con satisfaccion baja muestran un riesgo de salida claramente superior al promedio.",
            "Grafico de rotacion por satisfaccion.",
        ],
        [
            "Brecha de rotacion por satisfaccion",
            brecha_rotacion_satisfaccion,
            "p.p.",
            "Cuantifica el impacto de la satisfaccion en la salida y facilita priorizar intervenciones.",
            "H2.",
            "La distancia entre empleados con satisfaccion baja y alta confirma una brecha de riesgo relevante.",
            "Tarjeta ejecutiva y grafico de rotacion por satisfaccion.",
        ],
        [
            "Rotacion con salario bajo",
            rotacion_salario_bajo,
            "%",
            "Mide el riesgo de fuga en el segmento mas sensible a compensacion.",
            "H3.",
            "El salario bajo concentra una rotacion mayor que el promedio general.",
            "Tarjeta ejecutiva y grafico de rotacion por salario.",
        ],
        [
            "Brecha salario bajo vs alto",
            brecha_salario,
            "p.p.",
            "Dimensiona el costo relativo de no intervenir las diferencias de compensacion.",
            "H3.",
            "La diferencia de salida entre salario bajo y alto evidencia una brecha fuerte de retencion.",
            "Grafico de rotacion por salario.",
        ],
        [
            "Rotacion en alta carga laboral",
            rotacion_alta_carga,
            "%",
            "Activa una alerta operativa sobre equipos con exigencia alta aunque H4 no se confirme a nivel departamental.",
            "H1 y H4.",
            "La alta carga laboral concentra una salida elevada y debe monitorearse como senal preventiva.",
            "Grafico de rotacion por carga laboral.",
        ],
    ],
    columns=[
        "kpi",
        "valor",
        "unidad",
        "justificacion_negocio",
        "hipotesis_relacionadas",
        "interpretacion",
        "soporte_dashboard",
    ],
)

kpis.to_csv(DATA_DIR / "kpi_definitions.csv", index=False, encoding="utf-8")
display(kpis)


## Visualizaciones minimas

In [ ]:
# Preparamos la tabla para la grafica de rotacion por departamento.
department_metrics = (
    df.groupby("department_label", as_index=False)
    .agg(turnover_rate=("left", "mean"))
    .sort_values("turnover_rate", ascending=False)
)
department_metrics["turnover_rate_pct"] = department_metrics["turnover_rate"] * 100
display(department_metrics.head())

In [ ]:
# Grafico 1: departamentos con mayor y menor rotacion.
plt.figure(figsize=(9, 4))
sns.barplot(data=department_metrics, x="turnover_rate_pct", y="department_label", color="#3563a3")
for container in plt.gca().containers:
    plt.gca().bar_label(container, fmt="%.1f%%", padding=3, fontsize=8)
plt.title("Rotacion por departamento")
plt.xlabel("Rotacion (%)")
plt.ylabel("Departamento")
plt.tight_layout()
plt.show()


In [ ]:
# Grafico 2: relacion entre horas mensuales y satisfaccion.
# Usamos una muestra para que el grafico quede mas legible.
sample = df.sample(min(len(df), 3000), random_state=42)
corr_dashboard, p_dashboard = stats.pearsonr(df["average_monthly_hours"], df["satisfaction_level"])
plt.figure(figsize=(8, 5))
sns.regplot(
    data=sample,
    x="average_monthly_hours",
    y="satisfaction_level",
    scatter_kws={"alpha": 0.25, "s": 14, "color": "steelblue"},
    line_kws={"color": "red"},
)
plt.title("Horas mensuales vs satisfaccion")
plt.xlabel("Horas mensuales promedio")
plt.ylabel("Satisfaccion")
plt.annotate(
    f"r = {corr_dashboard:.2f} | p = {p_dashboard:.3f}",
    xy=(0.05, 0.95),
    xycoords="axes fraction",
    fontsize=10,
    va="top",
)
plt.tight_layout()
plt.savefig(VIS_DIR / "h1_horas_vs_satisfaccion.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Preparamos la tabla de porcentaje para la rotacion por satisfaccion.
satisfaction_plot = pd.crosstab(
    df["satisfaction_category"],
    df["left_label"],
    normalize="index",
).mul(100)
display(satisfaction_plot)

In [ ]:
# Grafico 3: compara permanencia y salida segun satisfaccion.
satisfaction_plot[["Permanece", "Se va"]].plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
    color=["steelblue", "tomato"],
)
for container in plt.gca().containers:
    plt.gca().bar_label(container, fmt="%.1f%%", label_type="center", fontsize=8)
plt.title("Rotacion por satisfaccion")
plt.xlabel("Categoria de satisfaccion")
plt.ylabel("Porcentaje (%)")
plt.xticks(rotation=0)
plt.legend(["Permanece (0)", "Se va (1)"])
plt.tight_layout()
plt.savefig(VIS_DIR / "h2_satisfaccion_vs_rotacion.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Preparamos la tabla de porcentaje para la rotacion por salario.
salary_plot = pd.crosstab(
    df["salary_label"],
    df["left_label"],
    normalize="index",
).mul(100)
display(salary_plot)

In [ ]:
# Grafico 4: compara permanencia y salida segun salario.
salary_plot.reindex(["Bajo", "Medio", "Alto"])[["Permanece", "Se va"]].plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
    color=["steelblue", "tomato"],
)
for container in plt.gca().containers:
    plt.gca().bar_label(container, fmt="%.1f%%", label_type="center", fontsize=8)
plt.title("Rotacion por salario")
plt.xlabel("Nivel salarial")
plt.ylabel("Porcentaje (%)")
plt.xticks(rotation=0)
plt.legend(["Permanece (0)", "Se va (1)"])
plt.tight_layout()
plt.savefig(VIS_DIR / "h3_salario_vs_rotacion.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Preparamos la tabla para la grafica de rotacion por carga laboral.
workload_metrics = (
    df.groupby("workload_category", as_index=False)
    .agg(turnover_rate=("left", "mean"))
)
workload_metrics["turnover_rate_pct"] = workload_metrics["turnover_rate"] * 100
workload_metrics["workload_category"] = pd.Categorical(
    workload_metrics["workload_category"],
    categories=["Baja carga", "Carga media", "Alta carga"],
    ordered=True,
)
workload_metrics = workload_metrics.sort_values("workload_category")
display(workload_metrics)

In [ ]:
# Grafico 5: resume la rotacion por carga laboral.
plt.figure(figsize=(7, 4))
sns.barplot(
    data=workload_metrics,
    x="workload_category",
    y="turnover_rate_pct",
    hue="workload_category",
    palette=["#b9852b", "#3563a3", "#c44e52"],
    legend=False,
)
for container in plt.gca().containers:
    plt.gca().bar_label(container, fmt="%.1f%%", padding=3, fontsize=8)
plt.title("Rotacion por carga laboral")
plt.xlabel("Carga laboral")
plt.ylabel("Rotacion (%)")
plt.tight_layout()
plt.show()


## Validacion final y trazabilidad con el dashboard


In [ ]:
# Confirmamos que el archivo obligatorio del dashboard exista.
print("Archivo obligatorio del dashboard:")
print(DASHBOARD_FILE)
print("Existe:", DASHBOARD_FILE.exists())

# Cerramos con un checklist minimo del PDF.
print("\nChecklist minimo del PDF:")
print(f"- KPIs de negocio definidos: {len(kpis)} en data/kpi_definitions.csv")
print("- 5 visualizaciones: listo en este notebook")
print("- Dashboard final: dashboard/hr_dashboard.pbix")
print("- Trazabilidad con dashboard: documentada en la columna soporte_dashboard")
print("- Visualizaciones guardadas: dashboard/visualizaciones/h1, h2 y h3")
